<a href="https://colab.research.google.com/github/seferyetzirah/python4me/blob/master/zipcode_polygon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Ian Saltzman 11/10/26. Polygons of student zip code data to congressional district.

import pandas as pd
import folium
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Load dataset from Google Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/studentcongress.csv'
df = pd.read_csv(file_path)

# 3. Aggregate unique student counts per Congressional District
district_summary = df.groupby('CongDistr')['State_StudentNumber'].nunique().to_dict()

# Calculate totals per district (pure + split counts)
pa02_total = district_summary.get('PA-02', 0) + district_summary.get('PA-02 (Split: PA-03)', 0)
pa03_total = district_summary.get('PA-03', 0) + district_summary.get('PA-03 (Split: PA-02)', 0) + district_summary.get('PA-03 (Split: PA-05)', 0)
pa05_total = district_summary.get('PA-05', 0) + district_summary.get('PA-05 (Split: PA-03)', 0)

# 4. District Information with Adjusted Centroids
district_info = {
    "PA-02": {
        "rep": "Brendan F. Boyle (D)",
        "count": pa02_total,
        "color": "#2563eb",
        "center": [40.0600, -75.0700]  # Centered over Northeast Philly
    },
    "PA-03": {
        "rep": "Dwight Evans (D)",
        "count": pa03_total,
        "color": "#16a34a",
        "center": [40.0100, -75.2100]  # Centered over West / Northwest Philly
    },
    "PA-05": {
        "rep": "Mary Gay Scanlon (D)",
        "count": pa05_total,
        "color": "#dc2626",
        "center": [39.9050, -75.1950]  # Centered over South Philly / Airport
    }
}

# 5. Define Inline GeoJSON Boundaries
geojson_shapes = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {"name": "PA-02", "geoid": "PA-02"},
      "geometry": {
        "type": "Polygon",
        "coordinates": [[
          [-75.14, 40.01], [-75.19, 40.06], [-75.07, 40.14], [-74.96, 40.14],
          [-74.98, 40.05], [-75.06, 40.01], [-75.14, 40.01]
        ]]
      }
    },
    {
      "type": "Feature",
      "properties": {"name": "PA-03", "geoid": "PA-03"},
      "geometry": {
        "type": "Polygon",
        "coordinates": [[
          [-75.28, 39.94], [-75.28, 40.08], [-75.19, 40.08], [-75.14, 40.01],
          [-75.14, 39.94], [-75.21, 39.94], [-75.28, 39.94]
        ]]
      }
    },
    {
      "type": "Feature",
      "properties": {"name": "PA-05", "geoid": "PA-05"},
      "geometry": {
        "type": "Polygon",
        "coordinates": [[
          [-75.25, 39.87], [-75.25, 39.94], [-75.14, 39.94], [-75.13, 39.88],
          [-75.20, 39.87], [-75.25, 39.87]
        ]]
      }
    }
  ]
}

# 6. Initialize Folium Map with CARTO API Key
carto_api_key = "cb1_56bj_1_a22b9e74afce5ef22ae91e97"
tile_url = f"https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={carto_api_key}"

m = folium.Map(
    location=[39.9800, -75.1400],
    zoom_start=11,
    tiles=tile_url,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'
)

# 7. Add District Polygon Layer
def style_function(feature):
    district_code = feature['properties']['name']
    info = district_info.get(district_code, {"color": "#6b7280"})
    return {
        'fillColor': info['color'],
        'color': info['color'],
        'weight': 2,
        'fillOpacity': 0.35
    }

def highlight_function(feature):
    return {'weight': 4, 'fillOpacity': 0.65}

folium.GeoJson(
    geojson_shapes,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=folium.GeoJsonTooltip(fields=['name'], aliases=['District:'])
).add_to(m)

# 8. Add Centered, Responsive Data Cards
for code, info in district_info.items():
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; width: 220px;">
        <h3 style="margin: 0 0 4px 0; color: {info['color']};">{code}</h3>
        <b>Representative:</b> {info['rep']}<hr style="margin: 6px 0;">
        <b>Total Student Count:</b> <span style="font-size: 14px; font-weight: bold;">{info['count']}</span>
    </div>
    """

    # Custom HTML Badge with auto-width and offset anchor
    label_html = f"""
    <div style="
        font-family: 'Segoe UI', Arial, sans-serif;
        font-size: 12px;
        font-weight: 700;
        color: {info['color']};
        background: #ffffff;
        padding: 6px 12px;
        border: 2px solid {info['color']};
        border-radius: 20px;
        box-shadow: 0 2px 6px rgba(0,0,0,0.25);
        white-space: nowrap;
        display: inline-block;
        transform: translate(-50%, -50%);
        cursor: pointer;
    ">
        {code}: {info['count']} Students
    </div>
    """

    folium.Marker(
        location=info['center'],
        icon=folium.DivIcon(
            html=label_html,
            icon_size=(0, 0),       # Let CSS auto-size width & height
            icon_anchor=(0, 0)      # Keep anchored to geographical center
        ),
        popup=folium.Popup(popup_html, max_width=250)
    ).add_to(m)

# 9. Render map
m

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
